### Source Tables:
- _exponent._bronze_allscripts_tw_works.dbo_item_result (measurement header - 413M records)
- _exponent._bronze_allscripts_tw_works.dbo_result (measurement values/results - 427M records)
- _exponent._bronze_allscripts_tw_works.dbo_item_finding (findings header - 283M records)
- _exponent._bronze_allscripts_tw_works.dbo_finding (finding values - 298M records)

### Concept Mapping:
- **NEW**: Uses `_exponent.results_store.omop_mapping_qo_de_measurement_final_output_v1`
  - Produced by `OMOP_mapping_notebook_script_measurement.ipynb`
  - Maps dbo_qo_de.ID (QODE) → OMOP concept_id via token matching
  - Contains: id, entryname, concept_id

### To Do:
- ~~Map QODE to OMOP measurement_concept_id using domain_source_to_concept~~
- ✅ Map QODE to measurement_concept_id using new mapping table
- Leverage LOINC codes from RIDLOINCCodeList for better concept mapping
- Map measurement_type_concept_id (default: 32817 = EHR)
- Link measurements to visits via ActivityHeaderID → order_activity_header → EncounterID
- Map unit_concept_id from UnitsDE/UnitsDET
- Parse reference ranges from ShortRefRange field
- Link to provider_id once provider table is populated

### Notes:
- PERSON and VISIT_OCCURRENCE must run before MEASUREMENT
- dbo_item_result.ID is the measurement identifier
- dbo_item_result.CurrentID links to dbo_result.ID for actual values
- dbo_item_result.PatientID links to dbo_person.ID
- Filters for NumericResult IS NOT NULL (quantitative data only)
- QODE = measurement type code (links to dbo_qo_de.ID)
- LOINC codes available in RIDLOINCCodeList column
- Starting with dbo_item_result/dbo_result (can add findings later)

# Transformation

In [0]:
%sql
TRUNCATE TABLE _exponent.omop_tw.measurement;

In [0]:
%sql
DELETE FROM _exponent.omop_silver.measurement
WHERE source_system = 'allscripts_tw';

In [0]:
%sql
DELETE FROM _exponent.omop_mapping.source_to_measurement
WHERE source_system = 'allscripts_tw';

In [ ]:
%sql
-- Using new measurement mapping table from scratch notebook:
-- _exponent.results_store.omop_mapping_qo_de_measurement_final_output_v1
-- Maps dbo_qo_de.ID (QODE) -> concept_id

CREATE OR REPLACE TEMPORARY VIEW silver_measurement AS 
SELECT 
  source_to_person.person_id,
  COALESCE(CAST(meas_map.concept_id AS BIGINT), 0) AS measurement_concept_id,
  CAST(COALESCE(r.ClinicalDTTM, ir.PerformedDTTM) AS DATE) AS measurement_date,
  COALESCE(r.ClinicalDTTM, ir.PerformedDTTM) AS measurement_datetime,
  NULL AS measurement_time,
  32817 AS measurement_type_concept_id,  -- EHR type concept (standard)
  NULL AS operator_concept_id,
  r.NumericResult AS value_as_number,
  NULL AS value_as_concept_id,
  0 AS unit_concept_id,
  NULL AS range_low,
  NULL AS range_high,
  NULL AS provider_id,
  source_to_visit_occurrence.visit_occurrence_id,
  NULL AS visit_detail_id,
  CONCAT_WS(
        CHR(31),
        'allscripts_tw',
        'dbo_item_result',
        'id',
        CAST(ir.ID AS BIGINT)
    ) AS measurement_source_value,
  0 AS measurement_source_concept_id,
  r.UnitsDET AS unit_source_value,
  0 AS unit_source_concept_id,
  CAST(r.NumericResult AS STRING) AS value_source_value,
  NULL AS measurement_event_id,
  NULL AS meas_event_field_concept_id,
  'allscripts_tw' AS source_system
FROM `_exponent`.`_bronze_allscripts_tw_works_vw`.`dbo_item_result` ir
INNER JOIN _exponent.omop_mapping.source_to_person
  ON CONCAT('allscripts_tw', CHAR(31), 'dbo_person', CHAR(31), 'id', CHAR(31), CAST(ir.PatientID AS BIGINT)) = source_to_person.person_source_value
  AND source_to_person.active_flag = TRUE
INNER JOIN `_exponent`.`_bronze_allscripts_tw_works_vw`.`dbo_result` r
  ON ir.CurrentID = r.ID
  AND r.NumericResult IS NOT NULL
LEFT JOIN `_exponent`.`_bronze_allscripts_tw_works_vw`.`dbo_order_activity_header` oah
  ON ir.ActivityHeaderID = oah.ID
LEFT JOIN _exponent.omop_mapping.source_to_visit_occurrence
  ON CONCAT('allscripts_tw', ' | ', oah.EncounterID) = source_to_visit_occurrence.visit_occurrence_source_value
  AND source_to_visit_occurrence.active_flag = TRUE
-- NEW: Join to measurement mapping table - FIX: Cast QODE to BIGINT first to remove decimals
LEFT JOIN _exponent.results_store.omop_mapping_qo_de_measurement_final_output_v1 meas_map
  ON CAST(CAST(ir.QODE AS BIGINT) AS STRING) = meas_map.id
WHERE ir.ID IS NOT NULL
  AND ir.PatientID IS NOT NULL
  AND ir.PerformedDTTM >= '2024-01-01'
  AND CAST(COALESCE(r.ClinicalDTTM, ir.PerformedDTTM) AS DATE) >= '1950-01-01'

In [ ]:
%sql
-- INSERT instead of MERGE for fresh runs (faster since we truncate first)
INSERT INTO _exponent.omop_silver.measurement (
  person_id,
  measurement_concept_id,
  measurement_date,
  measurement_datetime,
  measurement_time,
  measurement_type_concept_id,
  operator_concept_id,
  value_as_number,
  value_as_concept_id,
  unit_concept_id,
  range_low,
  range_high,
  provider_id,
  visit_occurrence_id,
  visit_detail_id,
  measurement_source_value,
  measurement_source_concept_id,
  unit_source_value,
  unit_source_concept_id,
  value_source_value,
  measurement_event_id,
  meas_event_field_concept_id,
  source_system
)
SELECT
  person_id,
  measurement_concept_id,
  measurement_date,
  measurement_datetime,
  measurement_time,
  measurement_type_concept_id,
  operator_concept_id,
  value_as_number,
  value_as_concept_id,
  unit_concept_id,
  range_low,
  range_high,
  provider_id,
  visit_occurrence_id,
  visit_detail_id,
  measurement_source_value,
  measurement_source_concept_id,
  unit_source_value,
  unit_source_concept_id,
  value_source_value,
  measurement_event_id,
  meas_event_field_concept_id,
  source_system
FROM silver_measurement

In [0]:
%sql
INSERT INTO _exponent.omop_mapping.source_to_measurement (
    source_system,
    measurement_source_value,
    person_id,
    active_flag,
    created_tsp,
    last_mod_tsp,
    merge_id,
    merge_reason
)
SELECT
    s.source_system,
    s.measurement_source_value,
    s.person_id,
    TRUE AS active_flag,
    current_timestamp() AS created_tsp,
    current_timestamp() AS last_mod_tsp,
    NULL AS merge_id,
    NULL AS merge_reason
FROM (
    SELECT DISTINCT 
        source_system, 
        measurement_source_value,
        person_id
    FROM _exponent.omop_silver.measurement
) s
LEFT ANTI JOIN _exponent.omop_mapping.source_to_measurement x
  ON s.measurement_source_value = x.measurement_source_value;

In [ ]:
%sql
-- INSERT instead of MERGE for fresh runs (faster since we truncate first)
INSERT INTO _exponent.omop_tw.measurement (
  measurement_id,
  person_id,
  measurement_concept_id,
  measurement_date,
  measurement_datetime,
  measurement_time,
  measurement_type_concept_id,
  operator_concept_id,
  value_as_number,
  value_as_concept_id,
  unit_concept_id,
  range_low,
  range_high,
  provider_id,
  visit_occurrence_id,
  visit_detail_id,
  measurement_source_value,
  measurement_source_concept_id,
  unit_source_value,
  unit_source_concept_id,
  value_source_value,
  measurement_event_id,
  meas_event_field_concept_id
)
SELECT
  source_to_measurement.measurement_id,
  s.person_id,
  s.measurement_concept_id,
  s.measurement_date,
  s.measurement_datetime,
  s.measurement_time,
  s.measurement_type_concept_id,
  s.operator_concept_id,
  s.value_as_number,
  s.value_as_concept_id,
  s.unit_concept_id,
  s.range_low,
  s.range_high,
  s.provider_id,
  s.visit_occurrence_id,
  s.visit_detail_id,
  s.measurement_source_value,
  s.measurement_source_concept_id,
  s.unit_source_value,
  s.unit_source_concept_id,
  s.value_source_value,
  s.measurement_event_id,
  s.meas_event_field_concept_id
FROM _exponent.omop_silver.measurement s
JOIN _exponent.omop_mapping.source_to_measurement
  ON source_to_measurement.measurement_source_value = s.measurement_source_value
  AND source_to_measurement.active_flag = TRUE
WHERE s.source_system = 'allscripts_tw'

In [0]:
%sql
-- Cleanup: Delete records with invalid dates (< 1950)
DELETE FROM _exponent.omop_tw.measurement
WHERE measurement_date < '1950-01-01';

In [0]:
%sql
-- Cleanup: Fix measurement_type_concept_id to valid EHR type concept
UPDATE _exponent.omop_tw.measurement
SET measurement_type_concept_id = 32817
WHERE measurement_type_concept_id <> 32817;

In [ ]:
%sql
-- Validation: Concept mapping coverage
SELECT 
  CASE WHEN measurement_concept_id = 0 THEN 'Unmapped' ELSE 'Mapped' END AS mapping_status,
  COUNT(*) AS record_count,
  ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER (), 2) AS pct
FROM _exponent.omop_tw.measurement
GROUP BY CASE WHEN measurement_concept_id = 0 THEN 'Unmapped' ELSE 'Mapped' END
ORDER BY record_count DESC